# RSNA Hemorrhage Sequence Classification — Colab Training

Clones the [rsna-hemorrhage-cv](https://github.com/ptuan21/rsna-hemorrhage-cv) repo, downloads the dataset from Kaggle ([samali012/rsna-multiwindow-sequence-384-v3](https://www.kaggle.com/datasets/samali012/rsna-multiwindow-sequence-384-v3)), and trains on the Colab GPU.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

In [ ]:
!git clone https://github.com/ptuan21/rsna-hemorrhage-cv.git
%cd rsna-hemorrhage-cv

In [ ]:
!pip install -q -r requirements.txt kaggle

## Kaggle credentials

Get your API token from https://www.kaggle.com/settings -> "Create New Token" (downloads `kaggle.json`), then run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select your kaggle.json here

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d samali012/rsna-multiwindow-sequence-384-v3 -p rsna_data --unzip

In [ ]:
# Sanity check: directory layout and split sizes should match the dataset README.
!ls rsna_data
!wc -l rsna_data/train.csv rsna_data/validation.csv rsna_data/test.csv

## Explore the data (EDA)

Class balance, sequence-length distribution, and what the multi-window slices actually look like, before spending GPU time training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LABELS = ["epidural", "subdural", "subarachnoid", "intraventricular", "intraparenchymal"]
CLASS_COLORS = {
    "epidural": "#2a78d6", "subdural": "#eb6834", "subarachnoid": "#1baf7a",
    "intraventricular": "#eda100", "intraparenchymal": "#e87ba4",
}
SPLIT_COLORS = {"train": "#2a78d6", "validation": "#eb6834", "test": "#1baf7a"}

splits = {name: pd.read_csv(f"rsna_data/{name}.csv") for name in ["train", "validation", "test"]}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sizes = [len(df) for df in splits.values()]
axes[0].bar(list(splits.keys()), sizes, color="#2a78d6", width=0.5)
for i, v in enumerate(sizes):
    axes[0].text(i, v + 50, f"{v:,}", ha="center", fontsize=10)
axes[0].set_title("Number of sequences per split")
axes[0].set_ylabel("Sequences")
axes[0].spines[["top", "right"]].set_visible(False)

x = np.arange(len(LABELS))
width = 0.25
for i, (name, df) in enumerate(splits.items()):
    counts = df["Label"].value_counts().reindex(LABELS).fillna(0)
    axes[1].bar(x + i * width, counts, width, label=name, color=SPLIT_COLORS[name])
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(LABELS, rotation=30, ha="right")
axes[1].set_title("Class distribution per split")
axes[1].legend(frameon=False)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

train_counts = splits["train"]["Label"].value_counts().reindex(LABELS)
print(
    f"Class imbalance (train): {train_counts.idxmax()} ({train_counts.max()}) is "
    f"{train_counts.max() / train_counts.min():.1f}x more common than "
    f"{train_counts.idxmin()} ({train_counts.min()})"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

train_df = splits["train"]
axes[0].hist(train_df["NumSlices"], bins=30, color="#2a78d6", edgecolor="white")
median = train_df["NumSlices"].median()
axes[0].axvline(median, color="#e34948", linestyle="--", linewidth=2, label=f"median = {median:.0f}")
axes[0].set_title("Slices per sequence (train)")
axes[0].set_xlabel("NumSlices")
axes[0].set_ylabel("Sequences")
axes[0].legend(frameon=False)
axes[0].spines[["top", "right"]].set_visible(False)

data_per_class = [train_df[train_df["Label"] == label]["NumSlices"] for label in LABELS]
bp = axes[1].boxplot(data_per_class, patch_artist=True)
for patch, label in zip(bp["boxes"], LABELS):
    patch.set_facecolor(CLASS_COLORS[label])
    patch.set_alpha(0.75)
axes[1].set_xticklabels(LABELS, rotation=30, ha="right")
axes[1].set_title("NumSlices distribution per class (train)")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(LABELS), figsize=(15, 3.8))
for ax, label in zip(axes, LABELS):
    row = train_df[train_df["Label"] == label].sample(1, random_state=0).iloc[0]
    seq = np.load(f"rsna_data/{row['NPYPath']}")["sequence"]
    mid = seq[seq.shape[0] // 2]
    ax.imshow(mid)
    ax.set_title(f"{label}\n({row['NumSlices']} slices)", fontsize=10)
    ax.axis("off")
fig.suptitle(
    "Example multi-window slice per hemorrhage subtype (R=Brain, G=Subdural, B=Bone)", y=1.04
)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

## Visualize test-set results

Confusion matrix and per-class F1 from `test_predictions.csv` (written by the evaluate cell above).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

preds_df = pd.read_csv("test_predictions.csv")
cm = confusion_matrix(preds_df["true_label"], preds_df["pred_label"], labels=LABELS)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(len(LABELS)))
axes[0].set_xticklabels(LABELS, rotation=45, ha="right")
axes[0].set_yticks(range(len(LABELS)))
axes[0].set_yticklabels(LABELS)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title("Confusion matrix (row-normalized)")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        color = "white" if cm_norm[i, j] > 0.5 else "black"
        axes[0].text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.0%}", ha="center", va="center", color=color, fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

f1s = f1_score(preds_df["true_label"], preds_df["pred_label"], labels=LABELS, average=None, zero_division=0)
colors = [CLASS_COLORS[l] for l in LABELS]
axes[1].bar(LABELS, f1s, color=colors)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("F1 score")
axes[1].set_title("Per-class F1 on test set")
axes[1].set_xticks(range(len(LABELS)))
axes[1].set_xticklabels(LABELS, rotation=30, ha="right")
for i, v in enumerate(f1s):
    axes[1].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Macro F1: {f1_score(preds_df['true_label'], preds_df['pred_label'], labels=LABELS, average='macro'):.4f}")

## Train

`--device auto` picks CUDA automatically on a Colab GPU runtime. The pipeline now includes:

- **Train-time augmentation** (flip, small rotation, brightness/contrast jitter) — sequence-consistent, disabled on val/test.
- **Bidirectional GRU** context over slice embeddings before attention pooling, so slice order/adjacency is no longer ignored.
- **Focal Loss** (`--focal-gamma`, default 2.0) + a **class-balanced sampler** to oversample rare classes like epidural.
- **Cosine LR annealing** over `--epochs`, **gradient clipping** (`--grad-clip-norm`, default 5.0), and **early stopping** (`--patience`, default 7 epochs with no val macro-F1 gain) — so it's safe to set `--epochs` generously.

If you hit `CUDA out of memory` on a smaller GPU, lower `--batch-size` and/or `--max-slices` (default 16).

In [ ]:
!python -m src.train --data-root rsna_data --checkpoint-dir checkpoints --epochs 40 --batch-size 8 --patience 7 --device auto

## Evaluate on the held-out test split

In [ ]:
!python -m src.evaluate --data-root rsna_data --split test --checkpoint checkpoints/best_model.pt --device auto --output-csv test_predictions.csv

## Save the trained checkpoint back to Google Drive (optional)

Colab runtimes are ephemeral — mount Drive and copy the checkpoint out before the session recycles.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/rsna-hemorrhage-cv
!cp checkpoints/best_model.pt test_predictions.csv /content/drive/MyDrive/rsna-hemorrhage-cv/